# Machine Learning Cloud Deployment Tutorial Series

Tutor: Daniel Ramandi
## Session 1: Introduction to architectures and Preparing ML Model

Machine learning does not stop at training a model. In practice, a model becomes useful only when it can be **packaged, hosted, and accessed by other people or systems**. In this tutorial series, we will go from a model trained locally on a laptop to a simple cloud-deployed application that can return predictions like a real product. 

**NOTE**: This notebook follows the ideas and structure from **[this material](https://pages.github.ubc.ca/ggeorg02/class_demos/docs/ml-deployment-optional/)** put together by **the amazing Gittu**!

## Learning objectives for the full tutorial series

By the end of this 3-session tutorial, you should be able to:

- explain the difference between **training**, **hosting**, and **serving** a machine learning model,
- compare common deployment architectures and understand their tradeoffs,
- describe the role of services such as **S3**, **Elastic Beanstalk**, **SageMaker**, **Lambda**, and **API Gateway**,
- set up the core AWS infrastructure needed for a deployment workflow,
- package a trained model for inference,
- deploy a simple application that sends user input to a hosted model and returns predictions.

## Learning objectives for this session

By the end of Session 1, you should be able to:

- describe the main components of a cloud ML system in plain language,
- compare several ways to structure a deployed ML product,
- understand why we focus on Elastic **Beanstalk + Sagemaker** (Approach C) in this tutorial,
- connect the architecture ideas in this session to the model you already trained in DSCI 571.

## The core deployment idea

At a high level, every ML deployment workflow is trying to solve the same problem:

1. a user or application sends some input,
2. a web application or API receives that input,
3. a model runs inference,
4. the prediction is returned.

What changes between architectures is **how we organize the pieces**.

> In other words: the goal is always the same, but the business can be organized in different ways.

## A burger business analogy

I came up with this example to make things slightly clearer. This might not be the best analogy, but give you a kickstart on understanding what it is that we are trying to do.

Imagine you are starting a burger business.

**Your goal**:  
a customer places an order, and your business returns a burger.

That is very similar to machine learning deployment:

- the **customer request** is the input data,
- the **burger** is the model prediction,
- and the **business setup** is the deployment architecture.

The big question is:

> **Where do we take the order, where do we cook the burger, and who hands the result back to the customer (and how)?**

That is exactly what changes between different ML deployment designs.

---

### Basic translation from burger language to ML language

- **Customer** → browser, app user, or API client  
- **Burger order** → input data  
- **Burger** → model prediction  
- **Cashier / waiter** → web app or API layer  
- **Chef/Burger recipe** → the model running inference  
- **Kitchen** → the compute environment where inference happens  
- **Storage room / pantry** → artifact storage  

## Four ways to run the burger business

### A. Small restaurant with its own kitchen  
The cashier takes the order, walks into the kitchen, the chef makes the burger, and the cashier gives it back.

This is the simplest setup: the **application and the model live together in one place**.

---

### B. Food truck with an on-call cook upstairs  
There is a small order window, but the cook is not waiting in the kitchen all day. When an order comes in, someone calls the cook, the burger gets made, and then the cook disappears again.

This is more **on-demand** and more **serverless**.

---

### C. Restaurant in front, professional central kitchen in back  
Customers place orders in a normal restaurant, but the burger is actually made in a separate specialized kitchen whose only job is cooking efficiently.

This is the architecture we focus on in this tutorial:
- the **app layer** and the **model layer** are separated,
- the design is cleaner,
- and it scales more naturally than putting everything in one place.

---

### D. Online ordering kiosk + dispatcher + professional kitchen  
Customers place orders through a kiosk, a dispatcher receives the order, sends it to the central kitchen, and returns the finished result.

This is the most decoupled setup, but also the most abstract for beginners.

## Why Approach C is a good balance

Approach C gives us a very useful middle ground.

On one side, putting the model directly inside the app is easy to understand, but the whole system becomes tightly coupled.  
On the other side, a fully serverless multi-component design is powerful, but it introduces more moving pieces very quickly.

Approach C keeps a familiar web app in front, while moving the model into its own dedicated hosting layer.

That means:

- the app can focus on talking to users,
- the model can focus on inference,
- and the two can evolve more independently.

> A good way to think about it:  
> the **restaurant handles customers**, and the **professional kitchen handles cooking**.

## AWS terminology in plain language

Before mapping the architectures, here are the main AWS services we will refer to:

- **S3**: storage for files such as model artifacts and deployment bundles  
- **EC2**: a virtual server in the cloud  
- **Elastic Beanstalk (EB)**: a managed way to run a web application on AWS  
- **SageMaker**: a managed service for machine learning workflows and hosted inference  
- **Lambda**: code that runs on demand without you managing a server directly  
- **API Gateway**: the public front door that receives API requests  
- **Flask**: the Python web framework used for the app layer in our tutorial  
- **Endpoint**: an address where a deployed service listens for requests  
- **IAM Role**: permissions that allow AWS services to talk to other AWS services securely

## Mapping AWS components to the burger business

| Burger idea | AWS / ML meaning |
|---|---|
| Customer | Browser, user, or API client |
| Cashier / waiter | Flask app |
| Restaurant building | Elastic Beanstalk environment |
| Storage room / pantry | S3 bucket |
| Chef | Trained model |
| Specialized kitchen | SageMaker endpoint |
| Order window / public front desk | API Gateway |
| On-call cook upstairs | Lambda |
| Delivery address for orders | Endpoint |
| Staff permission to access supplies or kitchen | IAM role |

## The four architectures in AWS terms

### Architecture A : EB + S3 + Flask  
**Burger version:** small restaurant with its own kitchen

- **Elastic Beanstalk** hosts the application
- **Flask** handles the UI/API
- the **model is loaded locally inside the app**
- **S3** stores the model artifact

This is simple, but the app and model are tightly coupled.

---

### Architecture B : Lambda + S3 + API Gateway  
**Burger version:** food truck with an on-call cook

- **API Gateway** receives the request
- **Lambda** runs the backend code when needed
- the **model is loaded locally inside Lambda**
- **S3** stores the model artifact

This reduces always-on infrastructure, but is less intuitive for beginners.

---

### Architecture C : EB + SageMaker + Flask  
**Burger version:** restaurant in front, professional kitchen in back

- **Elastic Beanstalk** hosts the application
- **Flask** handles the UI/API
- **SageMaker** hosts the model separately as an endpoint
- **S3** stores the model artifact

This is our main focus because it separates the app from the model while staying conceptually manageable.

---

### Architecture D : SageMaker + Lambda + API Gateway  
**Burger version:** online ordering kiosk + dispatcher + professional kitchen

- **API Gateway** is the public entry point
- **Lambda** handles request logic
- **SageMaker** hosts the model
- **S3** stores the model artifact

This is very modular, but introduces more moving parts.

## Key takeaway from Part 1

All deployment architectures try to solve the same problem:

> **input goes in, prediction comes out**

What changes is how the system is organized.

In this tutorial series, we focus on **Approach C** because it gives a nice balance between:

- simplicity,
- separation of responsibilities,
- and a more realistic cloud deployment pattern.

In the next part, you will start training a model that we will later deploy

# Part 2 : Training a deployable spam/ham model

In this part, we will take the core idea from DSCI 571 Lab 3 and turn it into a clean training pipeline that we can later deploy.

We will:
- load the SMS spam dataset,
- create a train/test split,
- train the best-performing model from the lab,
- evaluate it on held-out data,
- and save the fitted pipeline so it can be used later for inference.

> **Dataset:** Please download the SMS spam dataset from **[kaggle](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)** and save it as `spam.csv` inside a local `data/` folder.

In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Load and clean the dataset

The original CSV contains a few extra unnamed columns that we do not need. We will keep only:

- `target` → the label (`spam` or `ham`)
- `sms` → the raw text message

In [5]:
sms_df = pd.read_csv("data/spam.csv", encoding="latin-1")

sms_df = sms_df.drop(columns=["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"])
sms_df = sms_df.rename(columns={"v1": "target", "v2": "sms"})

sms_df.head()

,target,sms
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
sms_df["target"].value_counts()

target
ham     4825
spam     747
Name: count, dtype: int64

## 2. Train/test split

We will split the dataset into training and test sets.

The text column (`sms`) will be our input, and the label column (`target`) will be our output.

In [7]:
train_df, test_df = train_test_split(
    sms_df,
    test_size=0.2,
    random_state=123
)

X_train = train_df["sms"]
y_train = train_df["target"]

X_test = test_df["sms"]
y_test = test_df["target"]

print("Train size:", len(train_df))
print("Test size:", len(test_df))

Train size: 4457
Test size: 1115


## 3. Define the model pipeline

Instead of handling preprocessing and modeling separately, we will build a single **scikit-learn pipeline**.

This is useful because:
- it keeps preprocessing and prediction bundled together,
- it reduces mistakes at inference time,
- and it is much easier to save and deploy later.

Our pipeline has two steps:

1. **CountVectorizer**  
   converts text into numeric bag-of-words features

2. **SVC**  
   the classifier that predicts `spam` or `ham`

The hyperparameters below come from the best-performing model in the original lab workflow (based on what I have, feel free to do your own tuning if you like).

In [8]:
spam_clf = make_pipeline(
    CountVectorizer(
        stop_words="english",
        max_features=2500,
        binary=False
    ),
    SVC(
        C=298.4636456236878,
        gamma=0.001816926471964085,
        random_state=123
    )
)

spam_clf

,steps,"[('countvectorizer', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## 4. Fit the model

In [9]:
spam_clf.fit(X_train, y_train)

,steps,"[('countvectorizer', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


## 5. Evaluate on the test set

We now evaluate the fitted pipeline on held-out test data.

Because this is a spam detection problem with class imbalance, accuracy alone is not enough.  
So we also look at the full classification report.

In [10]:
y_pred = spam_clf.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:\n")
print(classification_report(y_test, y_pred))

Test accuracy: 0.9821

Classification report:

              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       982
        spam       0.96      0.89      0.92       133

    accuracy                           0.98      1115
   macro avg       0.97      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



This actually looks pretty good. Just a reminder that the dummy classifier in lab 3 was ~0.86 in test accuracy.

## 6. Try a few manual examples

Before saving the model, it is helpful to test a few messages directly. Try some of your own and see how good your model is doing!

In [13]:
sample_messages = [
    "Congratulations! You have won a free vacation. Reply now to claim your prize.",
    "Hey, are we still meeting at 3 pm?",
    "URGENT! Your account has been selected for a cash reward.",
    "Can you pick up milk on your way home?"
]

sample_preds = spam_clf.predict(sample_messages)

for msg, pred in zip(sample_messages, sample_preds):
    print(f"[{pred.upper()}] {msg}")

[SPAM] Congratulations! You have won a free vacation. Reply now to claim your prize.
[HAM] Hey, are we still meeting at 3 pm?
[SPAM] URGENT! Your account has been selected for a cash reward.
[HAM] Can you pick up milk on your way home?


## 7. Save the fitted pipeline for deployment

We will save:
- **the full fitted pipeline**: Make sure that your model object contain the fitted pipeline, and not just the model. Can you guess why this is important?
- **a small metadata file**: This is usefull for app development, because it contains what your labels are and what they mean.

This is what we will use later when packaging the model for deployment.

In [16]:
model_dir = Path("spam-deploy/model_build/model")
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(spam_clf, model_dir / "model.joblib")

metadata = {
    "model_name": "spam_ham_svc_pipeline",
    "labels": ["ham", "spam"],
    "input_type": "single text message",
    "notes": "CountVectorizer + SVC pipeline trained from DSCI 571 Lab 3 workflow"
}

with open(model_dir / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print("-", model_dir / "model.joblib")
print("-", model_dir / "metadata.json")

Saved:
- spam-deploy/model_build/model/model.joblib
- spam-deploy/model_build/model/metadata.json


## Project structure for deployment
Now that you have your model saved, let me explain why that path!

It is useful to organize the project in a way that matches the later deployment workflow.

We will use the following structure:

```text
spam-deploy/
├── model_build/
│   ├── train_model.ipynb
│   ├── inference.py
│   └── model/
│       ├── model.joblib
│       └── metadata.json
├── deploy_sagemaker.py
└── eb_app/
    ├── app.py
    ├── Procfile
    ├── requirements.txt
    └── templates/
        └── index.html

**Now you may ask **what are all those stuff**? A very valid technical question!**

`spam-deploy/`
- This is the main project folder for everything related to deployment.

`model_build/`
- This folder contains the model-side files:
  - the training notebook,
  - the SageMaker inference script,
  - and the saved model artifact.

`model_build/model/`
- This folder stores the files produced after training:
  - model.joblib → the fitted scikit-learn pipeline
  - metadata.json → useful information about labels and expected input

`model_build/inference.py`
- This is the script SageMaker will use later for loading the model, receiving input, running prediction, and returning output.
  - Don't worry, we will talk about it in the last session! 

`deploy_sagemaker.py`
- This script will later upload the model artifact to S3 and create the SageMaker endpoint. You'll understand what it means when we get there!

`eb_app/`
- This folder contains the web application that will run on Elastic Beanstalk.

`eb_app/app.py`
- The Flask application code.

`eb_app/Procfile`
- A small text file that tells Elastic Beanstalk how to start the app.

`eb_app/requirements.txt`
- The Python packages needed by the Flask application.

`eb_app/templates/index.html`
- The HTML page shown to the user.

**Keeping this structure from the beginning makes the later deployment steps much cleaner.**

Instead of moving files around at the last minute in session 3, we save the model directly into the location where the deployment workflow expects it. Now let's build the whole directory tree, with placeholder empty files!

In [18]:
Path("spam-deploy/eb_app/templates").mkdir(parents=True, exist_ok=True)

files_to_create = [
    "spam-deploy/model_build/inference.py",
    "spam-deploy/deploy_sagemaker.py",
    "spam-deploy/eb_app/app.py",
    "spam-deploy/eb_app/Procfile",
    "spam-deploy/eb_app/requirements.txt",
    "spam-deploy/eb_app/templates/index.html",
]

for file_path in files_to_create:
    Path(file_path).touch(exist_ok=True)

print("Project scaffold created successfully.")

Project scaffold created successfully.


## You're all set!

Now you have what you'd need for session 2!

I highly suggest you have everything ready so that you can follow the instructions in session 2. I will walk you through setting up all the infrastructure that are required, and there are many steps involved. As a result, it would be best if you have your laptop ready, aws logged in, and prepared for a marathon!

### Optional:

If you want, you can try to use a different model here. My suggestion is to have the `SVC` model ready as a back-up, but feel free to try a simple `RNN` (575) or using **word embedding models** (572) instead of `countvectorizer`. Note that you would need to later modify your inference.py, as well as the environment setup (requirements.txt) to make sure all the required libraries for transforming new input data are available.